# Notebook 05 — Item-Based Collaborative Filtering Retrieval

**Goal:** Build a personalized Item-CF retrieval model from historical user–item co-occurrence, compare it against Notebook 04 popularity using the **same warm-user future-view evaluation protocol**. We do **not** have impression logs; Recall/HitRate are offline implicit-feedback proxies.

**Important:** Fit separately using `train_history.csv` for validation (before Aug 1) and `test_history.csv` for the descriptive test snapshot (before Sep 1). Do not fit on future labels. Model selection must use validation only.

**Computational design:** Limit each historical user's co-occurrence basket to their most recent 20 distinct viewed items, cap high-frequency hub items for co-occurrence, and retain Top-50 similar items per source. This bounds computation on a laptop; it is an approximation, not full-catalog Item-CF.

## 1. Load inputs and confirm cutoff safety

In [1]:
from pathlib import Path
from collections import defaultdict
from itertools import combinations
import json, time
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

cwd = Path.cwd().resolve()
if (cwd / 'data' / 'processed').exists():
    ROOT = cwd
elif (cwd.parent / 'data' / 'processed').exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError(f'Cannot find data/processed from {cwd}')
PROCESSED = ROOT / 'data' / 'processed'
REPORTS = ROOT / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)

names = {
    'val_history': 'train_history.csv',
    'val_labels': 'validation_view_labels_warm_users.csv',
    'test_history': 'test_history.csv',
    'test_labels': 'test_view_labels_warm_users.csv',
}
d = {}
for key, filename in names.items():
    p = PROCESSED / filename
    assert p.is_file(), f'Missing {p}'
    d[key] = pd.read_csv(p)
    print(f'{key}: {len(d[key]):,} rows')

CUTOFFS = {'val': pd.Timestamp('2015-08-01', tz='UTC'),
           'test': pd.Timestamp('2015-09-01', tz='UTC')}
K_VALUES = [10, 20, 50, 100]
MAX_K = max(K_VALUES)
for split in ['val', 'test']:
    h, labels = d[f'{split}_history'], d[f'{split}_labels']
    assert h.timestamp.max() < int(CUTOFFS[split].timestamp() * 1000)
    assert labels.visitorid.isin(h.visitorid).all()
    assert not labels[['visitorid','itemid']].duplicated().any()
print('PASS: input and temporal checks')

val_history: 1,899,029 rows
val_labels: 28,634 rows
test_history: 2,452,391 rows
test_labels: 31,876 rows
PASS: input and temporal checks


## 2. Fit sparse Item-CF similarity

We build a binary user–item matrix from each user's **most recent 20 distinct views**, and compute sparse item co-occurrence `X.T @ X`. We exclude extremely popular hub items **only from similarity training** to avoid enormous co-occurrence blocks; these items remain eligible in the popularity fallback. Similarity is cosine-normalized co-occurrence, `cooc(i,j) / sqrt(freq(i)*freq(j))`.

The `MAX_TRAIN_ITEMS` limit and hub threshold are engineering safeguards. They must be reported with results; we can tune them on validation later. Memory may still be significant on some machines.

In [2]:
MAX_ITEMS_PER_USER = 20
MAX_ITEM_USER_FREQUENCY = 2000
TOP_NEIGHBORS = 50

def fit_itemcf(history):
    views = history.loc[history.event.eq('view'), ['visitorid','itemid','timestamp']]
    # Latest unique user-item interaction, then latest N items per user
    views = views.sort_values('timestamp', ascending=False, kind='mergesort')
    views = views.drop_duplicates(['visitorid','itemid'], keep='first')
    views = views.groupby('visitorid', sort=False).head(MAX_ITEMS_PER_USER)
    freq = views.itemid.value_counts()
    eligible_items = set(freq.index[freq <= MAX_ITEM_USER_FREQUENCY].astype(int))
    views = views.loc[views.itemid.isin(eligible_items), ['visitorid','itemid']]
    # A sparse matrix, never a dense users x items matrix
    user_ids, user_idx = np.unique(views.visitorid.to_numpy(), return_inverse=True)
    item_ids, item_idx = np.unique(views.itemid.to_numpy(), return_inverse=True)
    X = csr_matrix((np.ones(len(views), dtype=np.float32), (user_idx, item_idx)),
                   shape=(len(user_ids), len(item_ids)), dtype=np.float32)
    X.sum_duplicates()
    cooc = (X.T @ X).tocsr()
    cooc.setdiag(0)
    cooc.eliminate_zeros()
    degrees = np.asarray(X.sum(axis=0)).ravel()
    neighbors = {}
    for i in range(cooc.shape[0]):
        start, end = cooc.indptr[i], cooc.indptr[i + 1]
        indices = cooc.indices[start:end]
        if len(indices) == 0:
            continue
        scores = cooc.data[start:end] / np.sqrt(degrees[i] * degrees[indices])
        if len(indices) > TOP_NEIGHBORS:
            keep = np.argpartition(scores, -TOP_NEIGHBORS)[-TOP_NEIGHBORS:]
            indices, scores = indices[keep], scores[keep]
        order = np.lexsort((item_ids[indices], -scores))
        neighbors[int(item_ids[i])] = [(int(item_ids[indices[j]]), float(scores[j]))
                                        for j in order]
    diagnostics = {'training_users': int(X.shape[0]), 'training_items': int(X.shape[1]),
                   'binary_interactions': int(X.nnz), 'cooccurrence_nnz': int(cooc.nnz),
                   'items_with_neighbors': len(neighbors),
                   'hub_items_excluded_from_similarity': int((freq > MAX_ITEM_USER_FREQUENCY).sum())}
    return neighbors, diagnostics

models = {}
for split in ['val','test']:
    t0 = time.perf_counter()
    neighbors, diag = fit_itemcf(d[f'{split}_history'])
    models[split] = neighbors
    print(split, 'fit seconds:', round(time.perf_counter() - t0, 2), diag)

val fit seconds: 3.83 {'training_users': 960487, 'training_items': 195237, 'binary_interactions': 1393511, 'cooccurrence_nnz': 2203778, 'items_with_neighbors': 121194, 'hub_items_excluded_from_similarity': 0}
test fit seconds: 4.7 {'training_users': 1245795, 'training_items': 218183, 'binary_interactions': 1802185, 'cooccurrence_nnz': 2774062, 'items_with_neighbors': 139393, 'hub_items_excluded_from_similarity': 1}


## 3. Personalized candidate generation with popularity fallback

Use the user's **latest distinct viewed items** as seeds; score neighbor items by summed cosine similarity. Exclude **all historically interacted** items, as in Notebook 04. Fill remaining slots with train-history popularity. A user with no usable neighbors receives a pure popularity fallback. Scores and source attribution can later feed the ranking model.

In [3]:
SEED_ITEMS = 20

def fit_popularity(history):
    counts = history.loc[history.event.eq('view'), 'itemid'].value_counts()
    return (counts.rename_axis('itemid').reset_index(name='count')
            .sort_values(['count','itemid'], ascending=[False,True], kind='mergesort')
            .itemid.astype(int).tolist())

def user_history_maps(history, users):
    sub = history.loc[history.visitorid.isin(users), ['visitorid','itemid','event','timestamp']]
    seen = sub.groupby('visitorid')['itemid'].agg(set).to_dict()
    views = sub.loc[sub.event.eq('view')].sort_values('timestamp', ascending=False, kind='mergesort')
    views = views.drop_duplicates(['visitorid','itemid'], keep='first')
    seeds = views.groupby('visitorid', sort=False)['itemid'].agg(lambda x: x.head(SEED_ITEMS).tolist()).to_dict()
    return seen, seeds

def recommend_itemcf(user, seen_map, seeds_map, neighbors, popularity, k=100):
    seen = seen_map.get(user, set())
    scores = defaultdict(float)
    for seed in seeds_map.get(user, []):
        for item, similarity in neighbors.get(int(seed), []):
            if item not in seen:
                scores[item] += similarity
    # Deterministic ranking, then popularity fallback
    ranked = sorted(scores, key=lambda item: (-scores[item], item))
    recs = ranked[:k]
    used = set(recs)
    if len(recs) < k:
        for item in popularity:
            if item not in seen and item not in used:
                recs.append(item)
                used.add(item)
                if len(recs) == k:
                    break
    return recs, min(len(ranked), k)

results = {}
for split in ['val','test']:
    h, labels = d[f'{split}_history'], d[f'{split}_labels']
    users = sorted(int(x) for x in labels.visitorid.unique())
    seen, seeds = user_history_maps(h, users)
    popularity = fit_popularity(h)
    t0 = time.perf_counter()
    recs, personalized_counts = {}, []
    for user in users:
        recs[user], n_personalized = recommend_itemcf(user, seen, seeds, models[split], popularity, MAX_K)
        personalized_counts.append(n_personalized)
    elapsed = time.perf_counter() - t0
    assert all(len(v) == MAX_K and len(set(v)) == MAX_K for v in recs.values())
    assert all(not (set(v) & seen.get(u, set())) for u, v in recs.items())
    results[split] = {'recs': recs, 'seen': seen, 'users': users,
                      'elapsed_s': elapsed,
                      'users_with_cf_candidates': sum(x > 0 for x in personalized_counts),
                      'avg_cf_candidates': float(np.mean(personalized_counts))}
    print(f'{split}: {len(users):,} users, {elapsed:.2f}s batch generation, '
          f'{results[split]["users_with_cf_candidates"]:,} users with CF candidates, '
          f'{results[split]["avg_cf_candidates"]:.1f} CF candidates/user')
# Batch throughput is not online p95 latency.

val: 13,329 users, 0.67s batch generation, 12,024 users with CF candidates, 45.4 CF candidates/user
test: 15,137 users, 1.03s batch generation, 13,852 users with CF candidates, 47.1 CF candidates/user


## 4. Evaluate using the *exact same* Notebook 04 metrics

Primary denominator: **all warm-user future view pairs**, including cold-item and previously-seen positives. The policy's eligible-pair upper bound is reported separately. Macro Recall@K averages per-user recall; micro Recall@K pools positive pairs.

In [4]:
def evaluate(recs, labels, seen_map, history_items, k_values):
    positives = labels.groupby('visitorid')['itemid'].agg(set).to_dict()
    rows = []
    for k in k_values:
        recalls, hits, found_total, pos_total = [], [], 0, 0
        recommended_items = set()
        for user, truth in positives.items():
            recommended = set(recs[int(user)][:k])
            found = len(recommended & truth)
            recalls.append(found / len(truth))
            hits.append(float(found > 0))
            found_total += found
            pos_total += len(truth)
            recommended_items.update(recommended)
        rows.append({'K': k, 'Recall@K_macro': float(np.mean(recalls)),
                     'Recall@K_micro': found_total / pos_total,
                     'HitRate@K': float(np.mean(hits)),
                     'UniqueRecommendedItems': len(recommended_items),
                     'CatalogCoverage': len(recommended_items) / len(history_items)})
    eligible = sum(1 for u,i in labels[['visitorid','itemid']].itertuples(index=False, name=None)
                   if i in history_items and i not in seen_map.get(int(u), set()))
    return pd.DataFrame(rows), eligible

metrics = {}
for split in ['val','test']:
    m, eligible = evaluate(results[split]['recs'], d[f'{split}_labels'],
                           results[split]['seen'], set(d[f'{split}_history'].itemid.unique()), K_VALUES)
    metrics[split] = m
    results[split]['eligible_pairs'] = eligible
    print(f'\n{split.upper()}: {eligible:,}/{len(d[f"{split}_labels"]):,} eligible positive pairs')
    display(m.round(6))


VAL: 21,984/28,634 eligible positive pairs


,K,Recall@K_macro,Recall@K_micro,HitRate@K,UniqueRecommendedItems,CatalogCoverage
0,10,0.023008,0.021443,0.038187,48830,0.244530
1,20,0.033450,0.032269,0.054018,64720,0.324104
2,50,0.048697,0.049137,0.078025,77378,0.387493
3,100,0.061495,0.062373,0.096031,81055,0.405906



TEST: 24,875/31,876 eligible positive pairs


,K,Recall@K_macro,Recall@K_micro,HitRate@K,UniqueRecommendedItems,CatalogCoverage
0,10,0.022865,0.019921,0.036863,56331,0.252626
1,20,0.032339,0.028140,0.049944,75339,0.337870
2,50,0.048959,0.043544,0.072405,90838,0.407378
3,100,0.059683,0.054618,0.088591,94990,0.425999


## 5. Compare with Notebook 04 popularity baseline (validation is the selection split)

In [5]:
for split in ['val','test']:
    baseline_path = REPORTS / f'notebook04_{split}_popularity_metrics.csv'
    if baseline_path.is_file():
        baseline = pd.read_csv(baseline_path)
        comparison = metrics[split].merge(baseline, on='K', suffixes=('_ItemCF','_Popularity'))
        for col in ['Recall@K_macro', 'Recall@K_micro', 'HitRate@K']:
            comparison[f'{col}_delta'] = comparison[f'{col}_ItemCF'] - comparison[f'{col}_Popularity']
        print(f'\n{split.upper()} comparison (positive delta means Item-CF improved):')
        display(comparison[['K','Recall@K_micro_ItemCF','Recall@K_micro_Popularity',
                            'Recall@K_micro_delta','HitRate@K_delta','CatalogCoverage_ItemCF',
                            'CatalogCoverage_Popularity']].round(6))
    else:
        print('Missing popularity metrics:', baseline_path)


VAL comparison (positive delta means Item-CF improved):


,K,Recall@K_micro_ItemCF,Recall@K_micro_Popularity,Recall@K_micro_delta,HitRate@K_delta,CatalogCoverage_ItemCF,CatalogCoverage_Popularity
0,10,0.021443,0.002968,0.018475,0.032110,0.244530,0.000085
1,20,0.032269,0.005727,0.026542,0.043289,0.324104,0.000190
2,50,0.049137,0.011106,0.038032,0.058294,0.387493,0.000431
3,100,0.062373,0.017636,0.044737,0.065796,0.405906,0.000821



TEST comparison (positive delta means Item-CF improved):


,K,Recall@K_micro_ItemCF,Recall@K_micro_Popularity,Recall@K_micro_delta,HitRate@K_delta,CatalogCoverage_ItemCF,CatalogCoverage_Popularity
0,10,0.019921,0.002384,0.017537,0.031909,0.252626,0.000081
1,20,0.028140,0.005176,0.022964,0.039572,0.337870,0.000188
2,50,0.043544,0.011325,0.032219,0.051397,0.407378,0.000435
3,100,0.054618,0.017411,0.037207,0.057277,0.425999,0.000825


## 6. Save results, diagnostic configuration, and candidate samples

In [6]:
for split in ['val','test']:
    metrics[split].to_csv(REPORTS / f'notebook05_{split}_itemcf_metrics.csv', index=False)
    sample_users = results[split]['users'][:20]
    sample = pd.DataFrame([(u, rank, item) for u in sample_users
                           for rank,item in enumerate(results[split]['recs'][u], start=1)],
                          columns=['visitorid','rank','itemid'])
    sample.to_csv(REPORTS / f'notebook05_{split}_itemcf_sample.csv', index=False)

summary = {
    'method': 'itemcf_cosine_cooccurrence_with_popularity_fallback',
    'protocol': 'warm-user future-view pairs; exclude seen; cold/seen labels remain in denominator',
    'cutoffs_utc': {'validation':'2015-08-01', 'test':'2015-09-01'},
    'config': {'max_items_per_training_user': MAX_ITEMS_PER_USER,
               'max_item_user_frequency_for_similarity': MAX_ITEM_USER_FREQUENCY,
               'top_neighbors_per_item': TOP_NEIGHBORS, 'seed_items': SEED_ITEMS,
               'max_k': MAX_K},
    'validation': {'users':len(results['val']['users']), 'positive_pairs':len(d['val_labels']),
                   'eligible_pairs':results['val']['eligible_pairs'],
                   'batch_seconds':results['val']['elapsed_s'],
                   'users_with_cf_candidates':results['val']['users_with_cf_candidates'],
                   'avg_cf_candidates':results['val']['avg_cf_candidates'],
                   'metrics':metrics['val'].to_dict(orient='records')},
    'test': {'users':len(results['test']['users']), 'positive_pairs':len(d['test_labels']),
             'eligible_pairs':results['test']['eligible_pairs'],
             'batch_seconds':results['test']['elapsed_s'],
             'users_with_cf_candidates':results['test']['users_with_cf_candidates'],
             'avg_cf_candidates':results['test']['avg_cf_candidates'],
             'metrics':metrics['test'].to_dict(orient='records')},
}
out = REPORTS / 'notebook05_itemcf_retrieval.json'
out.write_text(json.dumps(summary, indent=2), encoding='utf-8')
assert all(results[s]['eligible_pairs'] <= len(d[f'{s}_labels']) for s in ['val','test'])
assert all(metrics[s]['Recall@K_micro'].between(0,1).all() for s in ['val','test'])
print('Saved:', out)
print('PASS: temporal cutoffs, warm users, candidate uniqueness/seen filtering, and metric exports')

Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\reports\notebook05_itemcf_retrieval.json
PASS: temporal cutoffs, warm users, candidate uniqueness/seen filtering, and metric exports


## 7. Interpretation and next stage

- Compare **validation Recall@50 / Recall@100** against Notebook 04. A personalized model is **not guaranteed** to beat popularity on sparse users.
- Report `users_with_cf_candidates`: it shows how often Item-CF adds personalized candidates beyond fallback.
- Very active users are truncated to 20 recent unique views for similarity construction; this is a computational tradeoff.
- Evaluation cohorts are defined using future labels; that is acceptable for **offline evaluation**, but production serving cannot know future-active users.
- **Next: Notebook 06 — Embedding Retrieval**, using train-only item/user representation learning.